# 04_baselines


*Notebook version of `src/04_baselines.py`: same code, split into cells. Running it does exactly what the script does.*

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page (MultiMediate): https://multimediate.perceptualui.org/datasets/Dataset_NoXi/ (if it is offline, use the links below)
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017: https://eprints.nottingham.ac.uk/45827 (DOI https://doi.org/10.1145/3136755.3136780)
- Extension NoXi+J (2024): https://arxiv.org/abs/2409.13726
- Data access: University of Augsburg, Human-Centered AI lab - noxi@hcai.eu
- Used here: 9 conversations (English 027, 028, 030; French 001, 013, 021; German 070, 072, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [ ]:
import os, sys
if os.getcwd().endswith("notebooks"):
    os.chdir("..")              # work from the project folder, like the scripts
sys.path.insert(0, "src")       # so helper files in src/ (e.g. splits.py) can be imported

Imports and settings.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

FEATURES = os.path.join("data", "features", "noxi_engagement.parquet")

Function `ccc`.

In [ ]:
def ccc(y_true, y_pred):
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    cov = ((yt - yt.mean()) * (yp - yp.mean())).mean()
    return (2 * cov) / (yt.var() + yp.var() + (yt.mean() - yp.mean()) ** 2 + 1e-9)

Function `main`.

In [ ]:
def main():
    # load the data
    df = pd.read_parquet(FEATURES)
    feat_cols = [c for c in df.columns if c.startswith(("voice_", "face_"))]
    X = df[feat_cols].fillna(0).to_numpy("float32")
    y = df["engagement"].to_numpy("float32")

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

    models = {
        "LinearRegression": LinearRegression(),
        "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        "GradientBoosting": HistGradientBoostingRegressor(max_iter=300, random_state=42),
        "XGBoost": XGBRegressor(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            tree_method="hist", random_state=42, n_jobs=-1,
        ),
    }

    print(f"{len(df)} windows, {len(feat_cols)} features\n")
    print(f"{'model':18s} {'CCC':>7} {'Pearson':>8} {'MAE':>7} {'R2':>7}", flush=True)

    # train each model, then score it on the unseen test data
    for name, m in models.items():
        pipe = make_pipeline(StandardScaler(), m)
        pipe.fit(Xtr, ytr)

        pred = pipe.predict(Xte)
        r = np.corrcoef(yte, pred)[0, 1]
        print(f"{name:18s} {ccc(yte, pred):7.3f} {r:8.3f} "
              f"{mean_absolute_error(yte, pred):7.3f} {r2_score(yte, pred):7.3f}", flush=True)

Run the step.

In [ ]:
if __name__ == "__main__":
    main()